# Etapa 1 · Datos — revisión remota de soldaduras

Este notebook:
1. Baja los datasets de **Roboflow**: radiografías de soldaduras y fotos de cordones.
2. Verifica que solo haya imágenes y etiquetas.
3. Convierte los recuadros a un formato común y unifica los nombres de los defectos.
4. Detecta copias aumentadas y fotos repetidas para que no haya fuga de datos.
5. Separa entrenamiento / validación / prueba por grupo.
6. Genera gráficos e imágenes con los recuadros dibujados (rojo = defecto, verde = bien).

**Antes de correr:** *Settings → Internet: On* y el secreto `ROBOFLOW_API_KEY` en *Add-ons → Secrets*.
Después: *Run All*. Los resultados quedan en la pestaña *Output* (`/kaggle/working`).

## 1. Instalación y código del proyecto

In [ ]:
!pip install -q roboflow imagehash
import os
os.makedirs("revision_pipes", exist_ok=True)
open("revision_pipes/__init__.py", "w").close()

In [ ]:
%%writefile revision_pipes/annotations.py
"""Formato común de anotaciones (recuadros) para todos los datasets.

Cada dataset trae los recuadros en un formato distinto:
- YOLO: un .txt por imagen, con valores relativos (0 a 1) al tamaño de la imagen.
- COCO: un .json con [x, y, ancho, alto] en píxeles.
- Polígonos (SWRD): lista de puntos que rodean el defecto.

Este módulo los lleva todos a `Box`, en píxeles absolutos (x_min, y_min, x_max, y_max),
para que el resto del proyecto no dependa del formato de origen.
"""

from __future__ import annotations

from collections.abc import Iterable, Sequence
from dataclasses import dataclass

MIN_BOX_SIDE = 1.0


class AnnotationError(ValueError):
    """Una anotación no se puede interpretar."""


@dataclass(frozen=True)
class Box:
    x_min: float
    y_min: float
    x_max: float
    y_max: float
    label: str

    @property
    def width(self) -> float:
        return self.x_max - self.x_min

    @property
    def height(self) -> float:
        return self.y_max - self.y_min

    @property
    def area(self) -> float:
        return max(self.width, 0.0) * max(self.height, 0.0)

    def clip(self, image_width: int, image_height: int) -> Box:
        """Recorta el recuadro para que no se salga de la imagen."""
        return Box(
            x_min=min(max(self.x_min, 0.0), image_width),
            y_min=min(max(self.y_min, 0.0), image_height),
            x_max=min(max(self.x_max, 0.0), image_width),
            y_max=min(max(self.y_max, 0.0), image_height),
            label=self.label,
        )

    def is_valid(self) -> bool:
        return self.width >= MIN_BOX_SIDE and self.height >= MIN_BOX_SIDE


def from_yolo(
    text: str, image_width: int, image_height: int, class_names: Sequence[str]
) -> list[Box]:
    """Lee un .txt de YOLO: `clase x_centro y_centro ancho alto` (relativos, 0 a 1)."""
    boxes = []
    for line_number, line in enumerate(text.splitlines(), start=1):
        parts = line.split()
        if not parts:
            continue
        if len(parts) != 5:
            raise AnnotationError(f"Línea {line_number}: se esperaban 5 valores, hay {len(parts)}.")
        try:
            class_id = int(parts[0])
            xc, yc, w, h = (float(value) for value in parts[1:])
        except ValueError as exc:
            raise AnnotationError(f"Línea {line_number}: valores no numéricos.") from exc
        if not 0 <= class_id < len(class_names):
            raise AnnotationError(f"Línea {line_number}: clase {class_id} desconocida.")
        boxes.append(
            Box(
                x_min=(xc - w / 2) * image_width,
                y_min=(yc - h / 2) * image_height,
                x_max=(xc + w / 2) * image_width,
                y_max=(yc + h / 2) * image_height,
                label=class_names[class_id],
            )
        )
    return boxes


def from_coco(coco: dict) -> dict[str, list[Box]]:
    """Lee un diccionario COCO y devuelve los recuadros agrupados por nombre de archivo."""
    try:
        categories = {c["id"]: c["name"] for c in coco["categories"]}
        images = {img["id"]: img["file_name"] for img in coco["images"]}
        result: dict[str, list[Box]] = {name: [] for name in images.values()}
        for ann in coco["annotations"]:
            x, y, w, h = ann["bbox"]
            result[images[ann["image_id"]]].append(
                Box(x, y, x + w, y + h, categories[ann["category_id"]])
            )
    except (KeyError, TypeError, ValueError) as exc:
        raise AnnotationError(f"Archivo COCO mal formado: {exc}") from exc
    return result


def from_polygon(points: Iterable[Sequence[float]], label: str) -> Box:
    """Convierte un polígono (lista de puntos x, y) en el recuadro que lo contiene."""
    points = list(points)
    if len(points) < 3:
        raise AnnotationError("Un polígono necesita al menos 3 puntos.")
    xs = [float(p[0]) for p in points]
    ys = [float(p[1]) for p in points]
    return Box(min(xs), min(ys), max(xs), max(ys), label)


def clean(boxes: Iterable[Box], image_width: int, image_height: int) -> list[Box]:
    """Recorta los recuadros a la imagen y descarta los que quedan sin tamaño."""
    clipped = (box.clip(image_width, image_height) for box in boxes)
    return [box for box in clipped if box.is_valid()]

In [ ]:
%%writefile revision_pipes/splits.py
"""Separación en entrenamiento, validación y prueba sin fuga de datos.

Si dos recortes salen de la misma placa radiográfica y uno cae en entrenamiento
y otro en prueba, el modelo "ya vio" esa placa y las métricas salen infladas.
Por eso se separa por grupo (la placa de origen), no por imagen suelta.
"""

from __future__ import annotations

import random
from collections.abc import Callable, Sequence
from typing import TypeVar

T = TypeVar("T")


def split_by_group(
    items: Sequence[T],
    group_of: Callable[[T], str],
    ratios: tuple[float, float, float] = (0.7, 0.15, 0.15),
    seed: int = 42,
) -> dict[str, list[T]]:
    """Reparte los grupos completos entre train, val y test según las proporciones.

    Es reproducible: con la misma semilla siempre da la misma separación.
    """
    if len(ratios) != 3 or any(r < 0 for r in ratios) or abs(sum(ratios) - 1.0) > 1e-6:
        raise ValueError("Las proporciones deben ser tres valores no negativos que sumen 1.")

    groups: dict[str, list[T]] = {}
    for item in items:
        groups.setdefault(group_of(item), []).append(item)

    names = sorted(groups)
    random.Random(seed).shuffle(names)

    n = len(names)
    n_train = round(n * ratios[0])
    n_val = round(n * ratios[1])
    assignment = {
        "train": names[:n_train],
        "val": names[n_train : n_train + n_val],
        "test": names[n_train + n_val :],
    }
    return {split: [item for g in gs for item in groups[g]] for split, gs in assignment.items()}

In [ ]:
%%writefile revision_pipes/taxonomy.py
"""Nombres comunes para los tipos de defecto en radiografías.

Cada dataset nombra los defectos a su manera ("Lack Of Penetration", "incomplete penetration",
"LOP", "Tungsten Inclusion" y "tungsten inclusion" en el mismo dataset...). Para entrenar un
solo modelo hace falta una lista única de clases, con nombres en español para la app.
"""

from __future__ import annotations

import re

# Nombre normalizado de entrada (minúsculas, sin signos) -> clase común
ALIASES = {
    "crack": "fisura",
    "cracks": "fisura",
    "porosity": "porosidad",
    "pore": "porosidad",
    "blowhole": "porosidad",
    "slag": "inclusion de escoria",
    "slag inclusion": "inclusion de escoria",
    "slag inclusions": "inclusion de escoria",
    "lack of fusion": "falta de fusion",
    "incomplete fusion": "falta de fusion",
    "lof": "falta de fusion",
    "lack of penetration": "falta de penetracion",
    "incomplete penetration": "falta de penetracion",
    "lop": "falta de penetracion",
    "non penetration": "falta de penetracion",
    "undercut": "socavadura",
    "undarcut": "socavadura",
    "burn through": "perforacion",
    "burnthrough": "perforacion",
    "tungsten inclusion": "inclusion de tungsteno",
    "excessive reinforcement": "sobremonta excesiva",
    "inadequate reinforcement": "sobremonta insuficiente",
    "root concavity": "concavidad de raiz",
    "good weld": "sin defecto",
    "no defect": "sin defecto",
}

COMMON_CLASSES = sorted(set(ALIASES.values()))


def _key(label: str) -> str:
    return re.sub(r"[^a-z]+", " ", label.lower()).strip()


def normalize_label(label: str) -> str:
    """Devuelve la clase común. Si el nombre no se conoce, devuelve 'otro: <original>'
    para que aparezca en el análisis y se pueda decidir qué hacer con él."""
    return ALIASES.get(_key(label), f"otro: {label.strip()}")

In [ ]:
%%writefile revision_pipes/visualize.py
"""Dibujo de recuadros sobre la imagen: rojo para defectos, verde para zonas sanas.

Es el componente visual del proyecto: subo una placa o una foto y veo
dónde está el problema y dónde está bien.
"""

from __future__ import annotations

from collections.abc import Iterable

from PIL import Image, ImageDraw

from revision_pipes.annotations import Box

RED = (220, 38, 38)
GREEN = (22, 163, 74)
AMBER = (217, 119, 6)

# Etiquetas que se consideran "sanas" (se dibujan en verde). El resto va en rojo.
GOOD_LABELS = {"good weld", "good_weld", "ok", "sin defecto", "no defect", "pipe"}
# Etiquetas que indican un problema general (cordón malo), en ámbar.
WARNING_LABELS = {"bad weld", "bad_weld"}


def color_for(label: str) -> tuple[int, int, int]:
    key = label.strip().lower()
    if key in GOOD_LABELS:
        return GREEN
    if key in WARNING_LABELS:
        return AMBER
    return RED


def draw_boxes(
    image: Image.Image, boxes: Iterable[Box], show_numbers: bool = True
) -> Image.Image:
    """Devuelve una copia de la imagen con los recuadros numerados y coloreados."""
    canvas = image.convert("RGB").copy()
    draw = ImageDraw.Draw(canvas)
    thickness = max(2, round(min(canvas.size) / 200))
    for number, box in enumerate(boxes, start=1):
        color = color_for(box.label)
        draw.rectangle(
            [box.x_min, box.y_min, box.x_max, box.y_max], outline=color, width=thickness
        )
        if show_numbers:
            tag = f"{number} {box.label}"
            left, top, right, bottom = draw.textbbox((box.x_min, box.y_min), tag)
            draw.rectangle([left - 2, top - 2, right + 2, bottom + 2], fill=color)
            draw.text((box.x_min, box.y_min), tag, fill=(255, 255, 255))
    return canvas


def summarize(boxes: Iterable[Box]) -> dict[str, int]:
    """Cuenta los recuadros por etiqueta (sirve también para contar caños)."""
    counts: dict[str, int] = {}
    for box in boxes:
        counts[box.label] = counts.get(box.label, 0) + 1
    return dict(sorted(counts.items()))

In [ ]:
import json
from collections import Counter
from pathlib import Path

import imagehash
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from revision_pipes.annotations import Box, clean, from_coco
from revision_pipes.splits import split_by_group
from revision_pipes.taxonomy import normalize_label
from revision_pipes.visualize import draw_boxes, summarize

WORK = Path("/kaggle/working")
DATA = Path("/tmp/data")  # los datos no van al Output, solo los resultados
DATA.mkdir(parents=True, exist_ok=True)
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
LABEL_EXT = {".json", ".txt", ".xml", ".csv", ".yaml", ".yml"}
DOC_EXT = {".md", ".pdf", ".docx"}

# nombre: (workspace, proyecto, versión o None = la última, tipo)
# Las versiones fijas son las que NO tienen aumentaciones (copias rotadas/recortadas).
SOURCES = {
    "rx_weld": ("kunkun-vhmx2", "weld-2z7bw", 1, "radiografias"),
    "rx_weld_data": ("vayun-goel-jlreu", "weld-data-4ln0h", None, "radiografias"),
    "cordones_ccby": ("computer-vision-5vq1v", "welding_defects-2tvbr", 1, "cordones"),
    "cordones_dominio_publico": ("computer-vision-pl2fz", "welding-defects-detection-ru1eb", None, "cordones"),
}

## 2. Descarga y verificación de seguridad

In [ ]:
def audit(folder: Path) -> Counter:
    """Cuenta extensiones y borra todo lo que no sea imagen, etiqueta o documentación."""
    found, removed = Counter(), []
    for path in folder.rglob("*"):
        if path.is_file():
            ext = path.suffix.lower()
            found[ext] += 1
            if ext not in IMAGE_EXT | LABEL_EXT | DOC_EXT:
                removed.append(str(path.relative_to(folder)))
                path.unlink()
    print("  Extensiones:", dict(found), "· descartados:", removed or "ninguno")
    return found


from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
for name, (workspace, project_id, version, kind) in SOURCES.items():
    project = rf.workspace(workspace).project(project_id)
    if version is None:
        try:
            version = max(int(str(v.version).split("/")[-1]) for v in project.versions())
        except Exception:
            version = 1
    print(f"{name} ({kind}) → versión {version}")
    project.version(version).download("coco", location=str(DATA / name))
    audit(DATA / name)

## 3. Lectura, nombres comunes, copias aumentadas y duplicados

In [ ]:
def load_coco_folder(folder: Path, normalize: bool) -> dict:
    result = {}
    for ann_file in folder.rglob("_annotations.coco.json"):
        boxes_by_name = from_coco(json.loads(ann_file.read_text()))
        for file_name, boxes in boxes_by_name.items():
            image_path = ann_file.parent / file_name
            if not image_path.exists():
                continue
            if normalize:
                boxes = [Box(b.x_min, b.y_min, b.x_max, b.y_max, normalize_label(b.label)) for b in boxes]
            with Image.open(image_path) as im:
                result[image_path] = clean(boxes, *im.size)
    return result


def original_name(path: Path) -> str:
    """Roboflow nombra las copias 'foto_jpg.rf.<hash>.jpg': el original es lo anterior a '.rf.'."""
    return path.name.split(".rf.")[0]


def group_images(paths: list) -> dict:
    """Agrupa copias aumentadas (mismo nombre original) y fotos casi idénticas (hash parecido)."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        parent[find(a)] = find(b)

    for path in paths:
        union(str(path), "orig:" + original_name(path))
    bits = []
    for path in paths:
        with Image.open(path) as im:
            bits.append(imagehash.phash(im.convert("L")).hash.flatten())
    bits = np.array(bits, dtype=bool)
    for i in range(len(paths)):
        distances = np.count_nonzero(bits[i + 1:] != bits[i], axis=1)
        for j in np.nonzero(distances <= 4)[0]:
            union(str(paths[i]), str(paths[i + 1 + j]))
    return {p: find(str(p)) for p in paths}


datasets, groups = {}, {}
for name, (_, _, _, kind) in SOURCES.items():
    loaded = load_coco_folder(DATA / name, normalize=(kind == "radiografias"))
    print(f"{name}: {len(loaded)} imágenes, {len({original_name(p) for p in loaded})} originales")
    datasets.setdefault(kind, {}).update(loaded)

for kind, items in datasets.items():
    groups[kind] = group_images(sorted(items))
    print(f"{kind}: {len(items)} imágenes · {len(set(groups[kind].values()))} fotos distintas")

## 4. Análisis exploratorio

In [ ]:
counts = {kind: Counter(b.label for boxes in items.values() for b in boxes) for kind, items in datasets.items()}
titles = {"radiografias": "Radiografías: defectos por tipo", "cordones": "Fotos de cordones: recuadros por clase"}
fig, axes = plt.subplots(1, len(counts), figsize=(7 * len(counts), 5), squeeze=False)
for ax, (kind, c) in zip(axes[0], counts.items()):
    labels, values = zip(*c.most_common())
    ax.barh(labels, values, color="#2563eb")
    ax.invert_yaxis()
    for i, v in enumerate(values):
        ax.text(v, i, f" {v}", va="center")
    ax.set_title(titles.get(kind, kind))
plt.tight_layout()
plt.savefig(WORK / "eda_clases.png", dpi=150)
plt.show()

summary = {
    kind: {
        "imagenes": len(items),
        "fotos_distintas": len(set(groups[kind].values())),
        "imagenes_sin_recuadros": sum(1 for b in items.values() if not b),
        "recuadros": sum(counts[kind].values()),
        "clases": dict(counts[kind].most_common()),
    }
    for kind, items in datasets.items()
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 5. Separación sin fuga de datos

In [ ]:
rows, summary_splits = [], {}
for kind, items in datasets.items():
    parts = split_by_group(sorted(items), lambda p, k=kind: groups[k][p])
    sets = {k: {groups[kind][p] for p in v} for k, v in parts.items()}
    assert not (sets["train"] & sets["val"] or sets["train"] & sets["test"] or sets["val"] & sets["test"])
    summary_splits[kind] = {k: len(v) for k, v in parts.items()}
    for split, paths in parts.items():
        print(f"{kind:12s} {split:5s}: {len(paths)} imágenes")
        rows += [f"{kind},{split},{p.relative_to(DATA)}" for p in paths]
print("✅ Sin fuga: ningún grupo aparece en dos conjuntos")
(WORK / "splits.csv").write_text("tipo,split,archivo\n" + "\n".join(rows), encoding="utf-8")
summary["splits"] = summary_splits
(WORK / "resumen_etapa1.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))

## 6. Ejemplos con recuadros (capturas para la bitácora)

In [ ]:
def to_display(im: Image.Image) -> Image.Image:
    """Las radiografías de 16 bits se ven negras o blancas si no se normalizan."""
    if im.mode in ("I;16", "I;16B", "I", "F"):
        a = np.asarray(im, dtype=np.float32)
        lo, hi = np.percentile(a, [1, 99])
        a = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
        return Image.fromarray((a * 255).astype("uint8"))
    return im


def pick_varied(items: dict, kind: str, per_class: int = 2, n: int = 6) -> list:
    chosen, seen, used_groups = [], Counter(), set()
    for p, b in sorted(items.items()):
        main = Counter(x.label for x in b).most_common(1)
        if main and seen[main[0][0]] < per_class and groups[kind][p] not in used_groups:
            chosen.append((p, b))
            seen[main[0][0]] += 1
            used_groups.add(groups[kind][p])
        if len(chosen) == n:
            break
    return chosen


def grid(items, title, out_name):
    if not items:
        print("Sin ejemplos para", title)
        return
    cols = 3
    rows_n = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(15, 5 * rows_n), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (path, boxes) in zip(axes.flat, items):
        with Image.open(path) as im:
            ax.imshow(draw_boxes(to_display(im), boxes))
        ax.set_title(", ".join(f"{k}: {v}" for k, v in summarize(boxes).items()), fontsize=10)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(WORK / out_name, dpi=130)
    plt.show()


grid(pick_varied(datasets.get("radiografias", {}), "radiografias"),
     "Radiografías: defectos marcados (rojo)", "ejemplos_radiografias.png")
grid(pick_varied(datasets.get("cordones", {}), "cordones"),
     "Fotos de cordones: rojo = defecto, ámbar = cordón malo, verde = cordón bueno", "ejemplos_cordones.png")
print("Listo. Descargá el notebook con File → Download Notebook y pasáselo a Claude.")